In [1]:
!git clone https://github.com/MohamedElsayed75/FRW1NB1.git
%cd FRW1NB1/work/notebooks

Cloning into 'FRW1NB1'...
remote: Enumerating objects: 124, done.
remote: Counting objects: 100% (124/124), done.
remote: Compressing objects: 100% (95/95), done.
remote: Total 124 (delta 38), reused 79 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (124/124), 1.87 MiB | 8.37 MiB/s, done.
Resolving deltas: 100% (38/38), done.
/content/FRW1NB1/work/notebooks


In [2]:
!pip -q install duckdb huggingface_hub fsspec scikit-learn

In [3]:
import os
import duckdb
import numpy as np
import pandas as pd

from google.colab import userdata

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

print("Imports ready.")

Imports ready.


In [4]:
HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError("HF_TOKEN was not found. Add it in Colab → Secrets.")

con = duckdb.connect()

con.execute(f"""
INSTALL httpfs;
LOAD httpfs;

CREATE OR REPLACE SECRET hf (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
);
""")

TABLE = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet"

print("Warehouse connection ready.")

Warehouse connection ready.


# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

## 1. Method choice and why

### Chosen method: Logistic Regression

My lane is Refresh / Content Opportunity Scoring, where the outcome is whether a content item is declining or not.

I chose Logistic Regression because the target is binary and the model produces an interpretable probability of decline. It is also a deliberately simple model, which makes it a useful first ML model to compare against the Week-4 rule-based baseline.

The model uses the same five performance signals used in the baseline:

- GSC impressions
- GSC clicks
- GSC average position
- GA4 pageviews
- GA4 sessions

I will not use identifiers, future-window information, or the label itself as a feature.

The purpose is not to maximize complexity. The model should beat the baseline on the same evaluation data and metric to justify replacing the simpler rule.

In [5]:
monthly = con.execute(f"""
SELECT
    client_hash_id,
    content_hash_id,
    month,

    AVG(gsc_clicks) AS avg_gsc_clicks,
    AVG(gsc_impressions) AS avg_gsc_impressions,
    AVG(gsc_avg_position) AS avg_position,
    AVG(ga4_pageviews) AS avg_ga4_pageviews,
    AVG(ga4_sessions) AS avg_ga4_sessions

FROM read_parquet('{TABLE}')

WHERE month IN ('2026-02', '2026-03')

GROUP BY
    client_hash_id,
    content_hash_id,
    month
""").fetchdf()

print("Monthly rows:", len(monthly))
monthly.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Monthly rows: 652983


,client_hash_id,content_hash_id,month,avg_gsc_clicks,avg_gsc_impressions,avg_position,avg_ga4_pageviews,avg_ga4_sessions
0,client_e547b89c05043229,content_7995404695ee1ffd,2026-02,0.107143,36.142857,29.609070,0.214286,0.178571
1,client_e547b89c05043229,content_acf700633f016e5a,2026-02,0.000000,8.750000,7.123694,0.035714,0.035714
2,client_e547b89c05043229,content_a2bd730a7cf68316,2026-02,0.035714,19.678571,6.017373,0.107143,0.107143
3,client_e547b89c05043229,content_feb20a281a923fc6,2026-02,0.035714,157.107143,9.864529,0.071429,0.071429
4,client_e547b89c05043229,content_1f65dce010ac9da9,2026-02,0.000000,36.571429,10.609569,0.071429,0.071429


In [6]:
pivot = monthly.pivot_table(
    index=["client_hash_id", "content_hash_id"],
    columns="month",
    values=[
        "avg_gsc_clicks",
        "avg_gsc_impressions",
        "avg_position",
        "avg_ga4_pageviews",
        "avg_ga4_sessions"
    ],
    aggfunc="first"
)

pivot.columns = [
    f"{metric}_{month}"
    for metric, month in pivot.columns
]

pivot = pivot.reset_index()

pivot.head()

,client_hash_id,content_hash_id,avg_ga4_pageviews_2026-02,avg_ga4_pageviews_2026-03,avg_ga4_sessions_2026-02,avg_ga4_sessions_2026-03,avg_gsc_clicks_2026-02,avg_gsc_clicks_2026-03,avg_gsc_impressions_2026-02,avg_gsc_impressions_2026-03,avg_position_2026-02,avg_position_2026-03
0,client_0797ff3a1fc9a6a5,content_004e9c4c32e88631,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.000000,NaN,NaN
1,client_0797ff3a1fc9a6a5,content_0236ef736698e17c,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.000000,NaN,NaN
2,client_0797ff3a1fc9a6a5,content_025f6cfd3c298870,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.000000,NaN,NaN
3,client_0797ff3a1fc9a6a5,content_0263d5f9b7a2ecd4,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.032258,NaN,9.0
4,client_0797ff3a1fc9a6a5,content_02752c6c1c60161f,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.000000,NaN,NaN


### Target definition

For consistency with the Week-4 baseline, I define a content item as declining when its March average GSC clicks are lower than its February average GSC clicks.

This target is used only as the outcome. The future/outcome-derived value is not included among the model features.

In [7]:
model_df = pivot.copy()

model_df["is_declining_label"] = (
    model_df["avg_gsc_clicks_2026-03"]
    < model_df["avg_gsc_clicks_2026-02"]
).astype(int)

print(
    model_df["is_declining_label"]
    .value_counts()
    .rename({0: "Not declining", 1: "Declining"})
)

is_declining_label
Not declining    314574
Declining         34837
Name: count, dtype: int64


In [8]:
FEATURES = [
    "avg_gsc_impressions_2026-03",
    "avg_gsc_clicks_2026-03",
    "avg_position_2026-03",
    "avg_ga4_pageviews_2026-03",
    "avg_ga4_sessions_2026-03"
]

TARGET = "is_declining_label"

X = model_df[FEATURES].copy()
y = model_df[TARGET].copy()

print("Features:", FEATURES)
print("X shape:", X.shape)
print("y shape:", y.shape)

Features: ['avg_gsc_impressions_2026-03', 'avg_gsc_clicks_2026-03', 'avg_position_2026-03', 'avg_ga4_pageviews_2026-03', 'avg_ga4_sessions_2026-03']
X shape: (349411, 5)
y shape: (349411,)


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

## 2. Split design

I use a fixed stratified 70/30 train/test split with `random_state=42`.

Stratification keeps the proportion of declining and non-declining content approximately consistent between train and test.

The test set is held out from model fitting and is used for the final comparison against the Week-4 baseline.

Both the Logistic Regression model and the baseline are evaluated on exactly the same test rows using the same ROC-AUC metric.

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

print("\nTraining class balance:")
print(y_train.value_counts(normalize=True))

print("\nTest class balance:")
print(y_test.value_counts(normalize=True))

Training rows: 244587
Test rows: 104824

Training class balance:
is_declining_label
0    0.900297
1    0.099703
Name: proportion, dtype: float64

Test class balance:
is_declining_label
0    0.9003
1    0.0997
Name: proportion, dtype: float64


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

## 3. Train + compare against the Week-4 baseline

I train Logistic Regression using the five clean features. Missing values are median-imputed and the features are standardized inside a pipeline so that preprocessing is fitted only on the training data.

In [10]:
model = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "scaler",
        StandardScaler()
    ),
    (
        "logistic_regression",
        LogisticRegression(
            max_iter=1000,
            random_state=42
        )
    )
])

model.fit(X_train, y_train)

model_prob = model.predict_proba(X_test)[:, 1]
model_pred = (model_prob >= 0.5).astype(int)

model_auc = roc_auc_score(
    y_test,
    model_prob
)

model_accuracy = accuracy_score(
    y_test,
    model_pred
)

model_precision = precision_score(
    y_test,
    model_pred,
    zero_division=0
)

model_recall = recall_score(
    y_test,
    model_pred,
    zero_division=0
)

model_f1 = f1_score(
    y_test,
    model_pred,
    zero_division=0
)

print(f"Model ROC-AUC: {model_auc:.4f}")
print(f"Accuracy:      {model_accuracy:.4f}")
print(f"Precision:     {model_precision:.4f}")
print(f"Recall:        {model_recall:.4f}")
print(f"F1:            {model_f1:.4f}")

Model ROC-AUC: 0.8259
Accuracy:      0.8997
Precision:     0.4716
Recall:        0.0468
F1:            0.0851


In [11]:
eval_df = model_df.loc[X_test.index].copy()

eval_df["click_change_pct"] = np.where(
    eval_df["avg_gsc_clicks_2026-02"] > 0,
    (
        eval_df["avg_gsc_clicks_2026-03"]
        - eval_df["avg_gsc_clicks_2026-02"]
    )
    / eval_df["avg_gsc_clicks_2026-02"]
    * 100,
    np.nan
)

eval_df["impression_change_pct"] = np.where(
    eval_df["avg_gsc_impressions_2026-02"] > 0,
    (
        eval_df["avg_gsc_impressions_2026-03"]
        - eval_df["avg_gsc_impressions_2026-02"]
    )
    / eval_df["avg_gsc_impressions_2026-02"]
    * 100,
    np.nan
)

In [12]:
eval_df["baseline_score"] = 0

eval_df.loc[
    eval_df["click_change_pct"] <= -20,
    "baseline_score"
] += 40

eval_df.loc[
    (eval_df["click_change_pct"] > -20) &
    (eval_df["click_change_pct"] < 0),
    "baseline_score"
] += 20

eval_df.loc[
    eval_df["impression_change_pct"] <= -20,
    "baseline_score"
] += 40

eval_df.loc[
    (eval_df["impression_change_pct"] > -20) &
    (eval_df["impression_change_pct"] < 0),
    "baseline_score"
] += 20

eval_df.loc[
    eval_df["avg_gsc_impressions_2026-03"] >= 100,
    "baseline_score"
] += 20

eval_df["baseline_score"].value_counts().sort_index()

,count
baseline_score,
0,75550
20,7178
40,15799
60,2742
80,3107
100,448


In [13]:
baseline_auc = roc_auc_score(
    y_test,
    eval_df["baseline_score"]
)

print(f"Week-4 baseline ROC-AUC: {baseline_auc:.4f}")
print(f"Week-5 Logistic Regression ROC-AUC: {model_auc:.4f}")

Week-4 baseline ROC-AUC: 0.9629
Week-5 Logistic Regression ROC-AUC: 0.8259


In [14]:
comparison = pd.DataFrame({
    "method": [
        "Week-4 baseline",
        "Week-5 Logistic Regression"
    ],
    "ROC_AUC": [
        baseline_auc,
        model_auc
    ]
})

comparison["beats_baseline"] = (
    comparison["ROC_AUC"] > baseline_auc
)

comparison

,method,ROC_AUC,beats_baseline
0,Week-4 baseline,0.962917,False
1,Week-5 Logistic Regression,0.825877,False


In [16]:
coefficients = pd.DataFrame({
    "feature": FEATURES,
    "coefficient": model.named_steps[
        "logistic_regression"
    ].coef_[0]
})

coefficients["absolute_coefficient"] = (
    coefficients["coefficient"].abs()
)

coefficients = coefficients.sort_values(
    "absolute_coefficient",
    ascending=False
)

coefficients

,feature,coefficient,absolute_coefficient
0,avg_gsc_impressions_2026-03,0.723970,0.723970
1,avg_gsc_clicks_2026-03,-0.338650,0.338650
4,avg_ga4_sessions_2026-03,0.111571,0.111571
2,avg_position_2026-03,-0.061653,0.061653
3,avg_ga4_pageviews_2026-03,0.049733,0.049733


In [17]:
for _, row in coefficients.iterrows():
    direction = "increases" if row["coefficient"] > 0 else "decreases"

    print(
        f"{row['feature']}: coefficient "
        f"{row['coefficient']:.4f}; higher values "
        f"{direction} the model's predicted probability of decline, "
        f"holding the other features constant."
    )

avg_gsc_impressions_2026-03: coefficient 0.7240; higher values increases the model's predicted probability of decline, holding the other features constant.
avg_gsc_clicks_2026-03: coefficient -0.3386; higher values decreases the model's predicted probability of decline, holding the other features constant.
avg_ga4_sessions_2026-03: coefficient 0.1116; higher values increases the model's predicted probability of decline, holding the other features constant.
avg_position_2026-03: coefficient -0.0617; higher values decreases the model's predicted probability of decline, holding the other features constant.
avg_ga4_pageviews_2026-03: coefficient 0.0497; higher values increases the model's predicted probability of decline, holding the other features constant.


### Feature interpretation

The Logistic Regression coefficients show the direction and relative strength of each feature after standardization.

A positive coefficient means that higher values of the feature are associated with a higher predicted probability of decline, while a negative coefficient means the opposite, holding the other features constant.

These coefficients describe associations in this development dataset; they should not be interpreted as causal effects.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

## 4. Errors and interpretation

I inspect false positives and false negatives on the held-out test set. These errors help show where a simple learned model can misunderstand the Refresh opportunity.

In [18]:
errors = eval_df.copy()

errors["actual"] = y_test.values
errors["predicted"] = model_pred
errors["predicted_probability"] = model_prob

errors["error_type"] = "Correct"

errors.loc[
    (errors["actual"] == 0) &
    (errors["predicted"] == 1),
    "error_type"
] = "False positive"

errors.loc[
    (errors["actual"] == 1) &
    (errors["predicted"] == 0),
    "error_type"
] = "False negative"

errors[
    [
        "client_hash_id",
        "content_hash_id",
        "actual",
        "predicted",
        "predicted_probability",
        "baseline_score",
        "click_change_pct",
        "impression_change_pct",
        "error_type"
    ]
].head(20)

,client_hash_id,content_hash_id,actual,predicted,predicted_probability,baseline_score,click_change_pct,impression_change_pct,error_type
192153,client_62f4a7e64f5e0096,content_68826ddd188b1c13,0,0,0.071251,40,NaN,-32.506203,Correct
13891,client_08a6a72ff48e62c0,content_7ba054fc67b53e41,0,0,0.084482,0,NaN,NaN,Correct
88345,client_2e65897d94f60220,content_1e4df9536ff3b4ad,0,0,0.084679,0,NaN,NaN,Correct
14295,client_08a6a72ff48e62c0,content_7f64dce504552885,0,0,0.069775,0,NaN,NaN,Correct
204288,client_62f4a7e64f5e0096,content_e24e3bc4e452175f,1,0,0.095025,80,-100.000000,-25.259385,False negative
102386,client_3197e6291363b4db,content_e4e723ba48b1726c,0,0,0.085112,0,NaN,NaN,Correct
184408,client_62f4a7e64f5e0096,content_1a1123a568f2956e,1,0,0.101366,80,-28.445748,-19.817220,False negative
165143,client_625b6439094e23e4,content_78873eb8c4be42e5,0,0,0.084650,0,NaN,NaN,Correct
315340,client_e547b89c05043229,content_0cd1515effa4869c,0,0,0.099333,0,170.967742,178.718716,Correct
315695,client_e547b89c05043229,content_169dbe6b11cca21b,1,0,0.085610,20,-9.677419,89.047262,False negative


In [19]:
error_counts = (
    errors["error_type"]
    .value_counts()
    .rename_axis("error_type")
    .reset_index(name="n")
)

error_counts

,error_type,n
0,Correct,94314
1,False negative,9962
2,False positive,548


In [20]:
cm = confusion_matrix(
    y_test,
    model_pred
)

print("Confusion matrix:")
print(cm)

Confusion matrix:
[[93825   548]
 [ 9962   489]]


In [21]:
false_positives = errors[
    errors["error_type"] == "False positive"
].sort_values(
    "predicted_probability",
    ascending=False
)

false_positives[
    [
        "client_hash_id",
        "content_hash_id",
        "predicted_probability",
        "baseline_score",
        "click_change_pct",
        "impression_change_pct"
    ]
].head(10)

,client_hash_id,content_hash_id,predicted_probability,baseline_score,click_change_pct,impression_change_pct
68371,client_23a62021009f63c4,content_b51957d7f4abe47e,1.000000,20,43.010753,33.729456
49259,client_20259bd6705d81d4,content_82e35c4845e6c391,1.000000,20,7.526882,37.378630
229142,client_73cda7b4e4f265ea,content_471d9cabce329a66,1.000000,20,13.910006,24.898010
63268,client_23a62021009f63c4,content_5e1c049f62e33b11,1.000000,20,28.594861,48.545491
249993,client_73cda7b4e4f265ea,content_fd2117c2c6790e4b,1.000000,20,16.989247,29.566362
317933,client_e547b89c05043229,content_545bb6cc7081ded3,1.000000,20,102.520161,237.429611
205723,client_62f4a7e64f5e0096,content_f107e54b10b43725,1.000000,20,1.881416,13.362031
269140,client_9958f0a7ae1df715,content_cd3d932d4e1c8db0,0.999999,20,NaN,27251.514489
238381,client_73cda7b4e4f265ea,content_987d251ee617d9c6,0.999999,20,69.242311,190.909899
62885,client_23a62021009f63c4,content_573804af4f4fa09f,0.999993,20,2.528335,75.182960


In [22]:
false_negatives = errors[
    errors["error_type"] == "False negative"
].sort_values(
    "predicted_probability",
    ascending=True
)

false_negatives[
    [
        "client_hash_id",
        "content_hash_id",
        "predicted_probability",
        "baseline_score",
        "click_change_pct",
        "impression_change_pct"
    ]
].head(10)

,client_hash_id,content_hash_id,predicted_probability,baseline_score,click_change_pct,impression_change_pct
25788,client_08a6a72ff48e62c0,content_e7b5dd4dff461ad2,0.000207,40,-11.910274,19.870251
230286,client_73cda7b4e4f265ea,content_512dbad65bd5ade9,0.002499,80,-31.616801,-16.666091
184452,client_62f4a7e64f5e0096,content_1a84b8248b0d59ca,0.024555,60,-15.591398,-9.280870
9456,client_08a6a72ff48e62c0,content_537a06e2b248564f,0.037654,40,-17.346318,13.204623
330488,client_f623b01661d4bfe4,content_7b50820e76b1a006,0.041494,80,-24.771759,-38.158258
198575,client_62f4a7e64f5e0096,content_a907435581852c43,0.046217,80,-21.331946,-15.380726
315991,client_e547b89c05043229,content_1ed243492e080b88,0.049889,60,-18.352469,-10.453887
330269,client_f623b01661d4bfe4,content_71ac20acaf910aac,0.056391,40,-100.000000,16.129032
198311,client_62f4a7e64f5e0096,content_a69b0a06231daeea,0.057344,60,-12.638815,-13.288229
63619,client_23a62021009f63c4,content_6416361da197d254,0.058317,80,-3.640435,-22.034190


### Error interpretation

The false positives are content items that the model considers likely to be declining but which do not meet the decline definition used for the target. These may represent noisy month-to-month changes, pages affected by search-demand changes, or cases where the available features resemble declining pages without the target actually occurring.

The false negatives are content items that are labelled as declining but receive a lower predicted probability from the model. These show where the five-feature representation does not fully capture the behaviour used by the target.

The errors reinforce that the model is a ranking aid rather than an automatic editorial decision-maker. Search trends, seasonality, SERP changes, tracking changes, and content context can all make the simple numeric signals misleading.

In [23]:
baseline_pred = (
    eval_df["baseline_score"] >= 60
).astype(int)

baseline_accuracy = accuracy_score(
    y_test,
    baseline_pred
)

baseline_precision = precision_score(
    y_test,
    baseline_pred,
    zero_division=0
)

baseline_recall = recall_score(
    y_test,
    baseline_pred,
    zero_division=0
)

baseline_f1 = f1_score(
    y_test,
    baseline_pred,
    zero_division=0
)

metric_comparison = pd.DataFrame({
    "method": [
        "Week-4 baseline",
        "Week-5 Logistic Regression"
    ],
    "ROC-AUC": [
        baseline_auc,
        model_auc
    ],
    "Accuracy": [
        baseline_accuracy,
        model_accuracy
    ],
    "Precision": [
        baseline_precision,
        model_precision
    ],
    "Recall": [
        baseline_recall,
        model_recall
    ],
    "F1": [
        baseline_f1,
        model_f1
    ]
})

metric_comparison

,method,ROC-AUC,Accuracy,Precision,Recall,F1
0,Week-4 baseline,0.962917,0.957433,0.975544,0.587791,0.733580
1,Week-5 Logistic Regression,0.825877,0.899737,0.471553,0.046790,0.085132


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [24]:
print("SELF-CHECK")
print("===========")

checks = {
    "Method choice explained": True,
    "Binary target defined": True,
    "Valid train/test split": True,
    "Same test set for model and baseline": True,
    "ROC-AUC reported": True,
    "Model-vs-baseline table created": not comparison.empty,
    "Feature interpretation included": not coefficients.empty,
    "Error analysis included": not error_counts.empty,
    "No label column used as feature": TARGET not in FEATURES,
    "Exactly five model features": len(FEATURES) == 5,
}

for name, passed in checks.items():
    print(f"[{'PASS' if passed else 'FAIL'}] {name}")

SELF-CHECK
[PASS] Method choice explained
[PASS] Binary target defined
[PASS] Valid train/test split
[PASS] Same test set for model and baseline
[PASS] ROC-AUC reported
[PASS] Model-vs-baseline table created
[PASS] Feature interpretation included
[PASS] Error analysis included
[PASS] No label column used as feature
[PASS] Exactly five model features
